In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report




print("Caricamento dati...")
train_df = pd.read_csv("development.csv")


train_df['title'] = train_df['title'].fillna('')
train_df['article'] = train_df['article'].fillna('')
train_df['text_content'] = train_df['title'] + " " + train_df['article']



text_transformer = TfidfVectorizer(
    stop_words='english',
    max_features=30000, 
    sublinear_tf=True,
    ngram_range=(1, 2) 
)

cat_transformer = OneHotEncoder(handle_unknown='ignore')
num_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('txt', text_transformer, 'text_content'),
        ('cat', cat_transformer, ['source']),
        ('num', num_transformer, ['page_rank'])
    ],
    remainder='drop'
)


classifier = LinearSVC(
    C=1.0, 
    random_state=42,
)

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', classifier)
])


print("Split Train/Validation interno...")
X = train_df[['text_content', 'source', 'page_rank']]
y = train_df['label']


param_grid = {
    'clf__C': [ 0.1, 1, 10],
}

grid_search = GridSearchCV(
    model_pipeline, 
    param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1, 
    verbose=2
)


grid_search.fit(X, y)

print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")


Caricamento dati...
Split Train/Validation interno...
Fitting 5 folds for each of 3 candidates, totalling 15 fits
{'clf__C': 0.1}
Best Cross-Validation Macro F1: 0.7148


In [4]:


best_model = grid_search.best_estimator_


eval_df = pd.read_csv("evaluation.csv")

eval_df['title'] = eval_df['title'].fillna('')
eval_df['article'] = eval_df['article'].fillna('')
eval_df['text_content'] = eval_df['title'] + " " + eval_df['article']

X_eval = eval_df[['text_content', 'source', 'page_rank']]


predictions = best_model.predict(X_eval)


submission_df = pd.DataFrame({
    'Id': eval_df['Id'],      
    'Predicted': predictions  
})


output_filename = "submission.csv"
submission_df.to_csv(output_filename, index=False)

print(f"File '{output_filename}' created!")
print(submission_df.head()) 

File 'submission.csv' created!
   Id  Predicted
0   0          5
1   1          2
2   2          0
3   3          0
4   4          0
